---
# <div style="text-align: center"> Azos Tutorial 1: Building Azos </div>
---

In this tutorial, you will **re-discover** one of the main pillars of SCOPE: the **System_azo** class.

*Recap from Tutorial 1...*

> *A **System** is a collection of chemical entities sharing common properties, typically their chemical composition. 
Its main purpose is to store paths for later use in SCOPE, as well as its **sources***.

Now, new classes are introduced, each as a subtype of the original SCOPE classes: 
- A **System_azo** is a specialized collection of chemical entities sharing something in common.   --> **Azobenzene**
- A **Molecule_azo** represents a Source from a System; it is an isolated chemical **specie** with specific features related to photochemistry.  --> **Isomers, Transition States...**
- The **Lamp** class is designed to **model** the behavior of a real light source, widely used in photochemistry. (See Tutorial 2: *Lights On: Reaching the Photostationary State*)

Throughout the previous tutorials, <span style="color:blue">**SCOPE**</span> has interacted with different parts of the code to handle the execution of computational workflows. 

Here are the topics covered in each Azo add-on tutorial:
1) New subclasses derived from CORE.
2) Running SCOPE tasks.
3) Post-calculation analysis. (WIP)

 
In the first part of this tutorial, we will create a **SYSTEM_AZO**, from its SMILES string, and learn about the creation of isomers, along with Transition States.  
In the second part, we will create a new system from scratch, and feed it with **sources**. 

## Part 1. Spinning up new Azo structures: Azobenzene

In [21]:
import os
import scope
import scope_azo

In [2]:
# First, let's define a SMILES string. It is important to be careful when 
#   constructing your SMILES: it must follow the pattern below to correctly detect
#   atom indices near the Azo group, enabling rotation and inversion of both rings.

#   IMPORTANT: SMILES strings MUST adhere to the following structure:
#                    'C1(...C1)/N=N/ring2' 
#                              or
#                    'C1(...C1)/N=N\\ring2' 


# We will select azobenzene for its simplicity. Let's store the SMILES string and its name.
 
smiles = 'c1(ccccc1)/N=N/c2ccccc2'
name = 'Azobenzene'


### The SYSTEM_AZO class: Where it all begins 

In [3]:
# We will create our system as a System_azo object providing its name and the smiles. 
azo_sys = scope_azo.System_azo(name, smiles)

In [4]:
# The SYSTEM_AZO class is a subtype of SYSTEM, as you can check:
print('Type:    ' ,azo_sys.type)
print('Subtype: ' ,azo_sys.subtype)

Type:     system
Subtype:  system_azo


In [5]:
# System_azo now possesses all the features of Systems. 
# You can confirm this by checking its _repr_ method:
azo_sys

# Note that a new attribute, 'dihedral_indices' is shown. 
# These will be relevant later.

------------- SCOPE System_azo Object -------
 Name                  = Azobenzene
 Version               = 1.0
 Type                  = system
 Subtype               = system_azo
 Dihedral Indices:     = [1, 0, 6, 7, 8, 9]
---------------------------------------------


### Working with System_azo Objects

Given a `system_azo` object, it's possible to obtain relevant structures such as **cis** and **trans** isomers. These functions from the `System_azo` class allow the creation of cis/trans isomers **and their transition state geometries**:

* **`System_azo.create_trans()`**
    * Creates **trans** structures.
    * It uses the SMILES string stored in the System to generate 3D coordinates.

* **`System_azo.create_cis(target_deg=40.)`**
    * Creates **cis** structures from the **trans** isomer.
    * *Note:* If the trans isomer has not been created before, it returns an error.
    * If everything works, it adds the cis isomer as a `Molecule_azo` object as source of the system.
    * The `target_deg` parameter allows you to choose the torsion of the dihedral angle of the Azo group (CN=NC). Please, insert your target angle in degrees.

* **`System_azo.create_ts(ts_list=['TSrot', 'TSinv_l', 'TSinv_r', 'triplet'])`**
    * The `ts_list` list parameter allows you to choose between which TS structures you want to create:
        * **`TSrot`**: Rotation of the azo group. Dihedral angles are set up as +90º (`TSrot_A`) and -90º (`TSrot_B`) in singlet state.
        * **`TSinv_l`**: Inversion of the left ring. The angle between CN=N atoms in the CN=NC azo group is set up as 180º.
        * **`TSinv_r`**: Inversion of the right ring. The angle between N=NC atoms in the CN=NC azo group is set up as 180º.
        * **`triplet`**: If this tag is selected, the `TSrot` are also created in the triplet state, by setting its spin as 2.

> You can read the docstring of each function for more information.

In [6]:
# If we attempt to create the cis isomer without first creating the trans isomer, it will raise an error.
azo_sys.create_cis()

Exception: AZO.CREATE_CIS: [ERROR] Trans isomer not found. Create it first with self.create_trans()

In [7]:
# Let's create the trans isomer. Note that the 'overwrite' parameter defaults to False.
trans_iso = azo_sys.create_trans(overwrite=True)
print(trans_iso)

---------- SCOPE Molecule_azo Object ------------
 Version               = 1.0
 Type                  = specie
 Sub-Type              = molecule_azo
 Name                  = trans
 Number of Atoms       = 24
 Formula               = H10-C12-N2
 Charge                = 0
 Spin (alpha - beta)   = 0
 SMILES                = c1(ccccc1)/N=N/c2ccccc2
 Number of Parents     = 0
 Has Adjacency Matrix  = NO 
 Has Bonds             = NO
-------------------------------------------------



In [8]:
# Species can be visualize using the method .view()
trans_iso.view(show_indices=True)

#Note: If you are running this on a remote cluster, the visualizer may take a while to load.

In [9]:
# Now we can generate the cis isomer. This returns a Molecule_azo object, which is automatically 
# registered as a source in our azobenzene system.
cis_iso = azo_sys.create_cis()
print(cis_iso)

---------- SCOPE Molecule_azo Object ------------
 Version               = 1.0
 Type                  = specie
 Sub-Type              = molecule_azo
 Name                  = cis
 Number of Atoms       = 24
 Formula               = H10-C12-N2
 Charge                = 0
 Spin (alpha - beta)   = 0
 SMILES                = c1(ccccc1)/N=N\c2ccccc2
 Number of Parents     = 0
 Has Adjacency Matrix  = NO 
 Has Bonds             = NO
-------------------------------------------------



In [10]:
# Finally, we can generate the Transition State (TS) geometries. 
# These are returned as a list and automatically registered as sources in the System_azo.

ts_list = azo_sys.create_ts()
print([ts.name for ts in ts_list])

# Pay attention to the naming convention: 
# Since all TSs are generated, the singlet and triplet states are labeled 
# 'TSrot_A_S' and 'TSrot_A_T', respectively.

['tsrot_a_s', 'tsrot_a_t', 'tsrot_b_s', 'tsrot_b_t', 'tsinv_l', 'tsinv_r']


In [11]:
# One way to recover sources is by its index, look at the indices above. For the cis isomer, the index is 1. 

cis = azo_sys.sources[1]
print(cis)

# Moreover, as Molecule_azo is a subclass of the MOLECULE/SPECIE class, we can use its functions, 
#   providing a great implementation with SCOPE features.
cis.view(show_indices=True)

---------- SCOPE Molecule_azo Object ------------
 Version               = 1.0
 Type                  = specie
 Sub-Type              = molecule_azo
 Name                  = cis
 Number of Atoms       = 24
 Formula               = H10-C12-N2
 Charge                = 0
 Spin (alpha - beta)   = 0
 SMILES                = c1(ccccc1)/N=N\c2ccccc2
 Number of Parents     = 0
 Has Adjacency Matrix  = NO 
 Has Bonds             = NO
-------------------------------------------------



In [12]:
# We can recover the source using their names. For instance, we can look for the one called 'trans'...
trans = azo_sys.find_source('trans')[1]
print(trans)

---------- SCOPE Molecule_azo Object ------------
 Version               = 1.0
 Type                  = specie
 Sub-Type              = molecule_azo
 Name                  = trans
 Number of Atoms       = 24
 Formula               = H10-C12-N2
 Charge                = 0
 Spin (alpha - beta)   = 0
 SMILES                = c1(ccccc1)/N=N/c2ccccc2
 Number of Parents     = 0
 Has Adjacency Matrix  = YES
 Has Bonds             = NO
-------------------------------------------------



In [13]:
# ... or one of the rotation TS in the triplet state 'TSrot_A_T'. 
ts = azo_sys.find_source('TSrot_a_t')[1]
print(ts)

---------- SCOPE Molecule_azo Object ------------
 Version               = 1.0
 Type                  = specie
 Sub-Type              = molecule_azo
 Name                  = tsrot_a_t
 Number of Atoms       = 24
 Formula               = H10-C12-N2
 Charge                = 0
 Spin (alpha - beta)   = 2
 Number of Parents     = 0
 Has Adjacency Matrix  = NO 
 Has Bonds             = NO
-------------------------------------------------



In [14]:
# The 3D structure of the models is created with Openbabel, and is accessible through:
print(trans.labels)
print(trans.coord)

['C', 'C', 'C', 'C', 'C', 'C', 'N', 'N', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H']
[[ 0.          0.          0.        ]
 [-0.51281336  1.30759191 -0.01964106]
 [-1.89451179  1.54886149 -0.01287176]
 [-2.79054639  0.48491126 -0.0356827 ]
 [-2.30684183 -0.820513   -0.04679195]
 [-0.92678473 -1.05609187 -0.02166651]
 [ 1.41048717 -0.16819286  0.08485943]
 [ 1.75056616 -1.37115871  0.13165836]
 [ 3.12815606 -1.57218314  0.36965153]
 [ 3.76079853 -2.64456496 -0.27758107]
 [ 5.08602564 -2.98826028  0.01472445]
 [ 5.81201493 -2.23890833  0.93462448]
 [ 5.20425284 -1.16482726  1.58394187]
 [ 3.86719973 -0.84465022  1.312251  ]
 [ 0.17005349  2.15269137 -0.02956064]
 [-2.26985447  2.56978884  0.01876031]
 [-3.8625511   0.66852757 -0.04428309]
 [-2.99892996 -1.65771281 -0.06143462]
 [-0.58140473 -2.0871111  -0.00947184]
 [ 3.2081124  -3.22976847 -1.0071315 ]
 [ 5.54380937 -3.84906914 -0.46783483]
 [ 6.84682129 -2.4945858   1.15225226]
 [ 5.76428278 -0.5887

In [15]:
# These Molecules_azo store labels and coordinates of the geometry stored in the 'initial' state.
import numpy as np

initial_state = trans.find_state('initial')[1]
print('Coordinates are equal: ', np.array_equal(initial_state.coord,trans.coord))

Coordinates are equal:  True


In [16]:
# Let's inspect the relevant atom indices for the trans isomer by accessing the 
# 'dihedral_indices' attribute:

print('Atom Indices for Dihedral: ', azo_sys.dihedral_indices)

# Recall that the atoms involved in the C-N=N-C rotation correspond to indices 0, 6, 7, and 8.
# These represent the four atoms extracted from the index list defining the torsion.

# The first and last elements of this list correspond to the anchor atoms of the 
# left and right rings, respectively.

# The variables 'atom0' and 'atom1' will be used to facilitate independent ring rotation.

atom0, atom1, atom2, atom3, atom4, atom5 = azo_sys.dihedral_indices

Atom Indices for Dihedral:  [1, 0, 6, 7, 8, 9]


In [23]:
# These indices allow us to analyze geometric parameters, such as the dihedral angle defined by the C-N=N-C ...
coord = trans.coord
dih_angle = np.degrees(scope.geometry.get_dihedral(coord[atom1], coord[atom2], coord[atom3], coord[atom4]))
print(f'{dih_angle:.2f} degrees')

173.14 degrees


In [24]:
# ... or the angle between CNN atoms. Here is the angle defined by the CN=N atoms. 
angle = np.degrees(scope.geometry.get_angle(coord[atom1] - coord[atom2], coord[atom2] - coord[atom3]))
print(f'{angle:.2f} degrees')

67.33 degrees


In [26]:
# Using the core SCOPE module, modifying dihedral angles is straightforward:

# (Caution: Rotations may lead to steric clashes. See the 'Advanced Users'
#  section at the end of this Notebook for details.)

rotated_coord = scope.geometry.set_dihedral(trans.labels, coord, 30, atom1, atom2, atom3, atom4)
new_molec     = scope_azo.Molecule_azo(trans.labels, rotated_coord)
new_molec.view(show_indices=True)

# Notice the steric clashes involving atoms 5, 13, 18, and 23. 
# Don't worry—we have a specific solution to resolve this.

In [ ]:
# The create_cis() method automatically resolves steric clashes when rotating 
# to a specific angle, defined by the 'target_deg' parameter.

# We recommend using this function for geometries prone to collisions (steric hindrance),
# as it automatically adjusts adjacent rings to achieve a valid, clash-free conformation.
azo_sys.create_cis(target_deg=30, overwrite=True).view()

> **Pro Tip:** We recommend using `create_cis()` for geometries prone to steric hindrance. It automatically adjusts adjacent rings to ensure a valid, clash-free conformation.

### Set Paths and calculations 

In [27]:
# Now, let's configure the directory paths. 
# You can use the interactive azo_sys.set_paths() method (ideal for terminal sessions) 
# or define them manually. Below, we demonstrate the explicit approach.
 
tutorial_folder = os.path.abspath('../')+'/Data/Azo/'
print(tutorial_folder)

# Defining paths explicitly:
azo_sys.set_main_path(f"{tutorial_folder}")
azo_sys

/Users/sergivela/Documents/SCOPE/Program/Scope_Tutorials/Data/Azo/


------------- SCOPE System_azo Object -------
 Name                  = Azobenzene
 Version               = 1.0
 Type                  = system
 Subtype               = system_azo
 Source Path           = /Users/sergivela/Documents/SCOPE/Program/Scope_Tutorials/Data/Azo/Sources/Azobenzene/
 System File Path      = /Users/sergivela/Documents/SCOPE/Program/Scope_Tutorials/Data/Azo/Systems/Azobenzene/
 System File Name      = /Users/sergivela/Documents/SCOPE/Program/Scope_Tutorials/Data/Azo/Systems/Azobenzene/Azobenzene.npy
 Computations Path     = /Users/sergivela/Documents/SCOPE/Program/Scope_Tutorials/Data/Azo/Computations/Azobenzene/

 Num Sources           = 8
     idx: type, name, formula               
     0: specie, trans, H10-C12-N2 
     1: specie, cis, H10-C12-N2 
     2: specie, tsrot_a_s, H10-C12-N2 
     3: specie, tsrot_a_t, H10-C12-N2 
     4: specie, tsrot_b_s, H10-C12-N2 
     5: specie, tsrot_b_t, H10-C12-N2 
     6: specie, tsinv_l, H10-C12-N2 
     7: specie, tsinv_r,

In [29]:
# The system can be saved to a .npy binary using 'load_binary()' function.
# As we have just assigned the system filepath, saving the binary becomes easy.

scope.save_binary(azo_sys, azo_sys.system_file)

## Part 2. Going further: Combining smiles. (WIP)

In [ ]:
# In computational studies, it is common to study multiple systems simultaneously.

# To speed up this, we allow you to construct systems by combining specific ring structures.
# You simply need to provide the SMILES for each ring (including substituents), adhering to 
# the pattern shown above:
#                           C1(...C1)/N=N/ring2 

# From now on, we will refer to these rings as 'fragments'.

# As an example, let's define two fragments for the left ring and one for the right.
# IMPORTANT: Each fragment MUST be a string stored within a list.

lefts   = ['C1=(C-C=C-C=C1)','C1=(CC=NC=C1)']
rights  = ['c2ccccc2']



In [ ]:
# The combine_smiles() function returns a list containing every combination of the fragments. 
# You can optionally provide a filename to save the output to a text file.

combine_smiles(lefts,rights)

# NO FUNCIONA ENCARA 

TypeError: combine_smiles() missing 1 required positional argument: 'subs'

In [ ]:
azoben_sys